In [ ]:
# Install EasyDiffraction and prepare the example when running in Colab.

import html
import importlib
import importlib.metadata
import importlib.util
import json
import os
import re
import subprocess
import sys
import urllib.parse
import urllib.request

RESTART = 'edi is already imported in this kernel. Restart the runtime, then run this cell again.'


def install_edi(index: str) -> None:
    """Install the wheel EasyDiffraction's index lists, and admit ``import edi`` only for it.

    The wheel is installed by its URL and sha256 with pip's settings ignored, so another index or
    pip configuration cannot supply it, and an unreachable index stops here. Its dependencies are
    installed on every run, by name, with the usual pip settings. If edi is already imported in
    this kernel, setup installs what is needed and asks for a restart; it never vouches for a
    loaded module. Otherwise ``import edi`` is admitted when the file it would load is one the
    installed wheel's RECORD lists. An edi no pip distribution owns is a source build, used as is.
    """
    owned = bool(importlib.metadata.packages_distributions().get('edi'))
    if 'edi' in sys.modules and not owned:
        raise RuntimeError(RESTART)
    spec = None if 'edi' in sys.modules else importlib.util.find_spec('edi')
    if spec is not None and not owned:
        print('Using edi from', spec.origin, '(a source build, not installed by pip)')
        return
    url, digest = _wheel(index)
    if not _installed_from(url, digest):
        _pip('--no-deps', '--force-reinstall', f'easydiffraction @ {url}#sha256={digest}')
    importlib.invalidate_caches()
    requires = importlib.metadata.requires('easydiffraction') or []
    if requires:
        _pip(*requires, settings=True)
    if 'edi' in sys.modules:
        raise RuntimeError(RESTART)
    importlib.invalidate_caches()
    if not _admitted(url, digest):
        raise RuntimeError('import edi would not load the wheel from ' + index)


def _wheel(index: str) -> tuple[str, str]:
    """The newest wheel on our index that this interpreter can install, with its sha256."""
    try:
        from packaging.tags import sys_tags  # noqa: PLC0415 - optional in a notebook
    except ImportError:
        from pip._vendor.packaging.tags import sys_tags  # noqa: PLC0415, PLC2701 - pip's copy
    supported = [str(tag) for tag in sys_tags()]
    page_url = urllib.parse.urljoin(index.rstrip('/') + '/', 'easydiffraction/')
    # A web index serves the folder's page itself; a local one is read from its index.html.
    page_file = page_url + 'index.html' if page_url.startswith('file:') else page_url
    with urllib.request.urlopen(page_file, timeout=60) as response:  # noqa: S310 - our index URL
        page = response.read().decode('utf-8')
    found = []
    for link in re.findall(r'href="([^"]+)"', page):
        url, _, fragment = urllib.parse.urljoin(page_url, html.unescape(link)).partition('#')
        name = url.rsplit('/', 1)[-1]
        match = re.fullmatch(r'easydiffraction-([0-9.]+)-([^-]+)-([^-]+)-([^-]+)\.whl', name)
        allowed = urllib.parse.urlparse(url).scheme in {
            'https',
            urllib.parse.urlparse(index).scheme,
        }
        if not match or not fragment.startswith('sha256=') or not allowed:
            continue
        tags = {
            f'{i}-{a}-{p}'
            for i in match[2].split('.')
            for a in match[3].split('.')
            for p in match[4].split('.')
        }
        ranks = [supported.index(tag) for tag in tags if tag in supported]
        if ranks:
            version = tuple(int(part) for part in match[1].split('.'))
            found.append((version, -min(ranks), url, fragment.removeprefix('sha256=')))
    if not found:
        raise RuntimeError('No EasyDiffraction wheel for this Python on ' + page_url)
    _, _, url, digest = max(found)
    return url, digest


def _installed_from(url: str, digest: str) -> bool:
    """Whether the installed easydiffraction came from this wheel URL and sha256."""
    try:
        dist = importlib.metadata.distribution('easydiffraction')
    except importlib.metadata.PackageNotFoundError:
        return False
    info = json.loads(dist.read_text('direct_url.json') or '{}')
    archive = info.get('archive_info', {})
    hashes = (
        archive.get('hashes', {}).get('sha256'),
        archive.get('hash', '').removeprefix('sha256='),
    )
    return info.get('url') == url and digest in hashes


def _admitted(url: str, digest: str) -> bool:
    """Whether the file ``import edi`` would load is one the installed wheel's RECORD lists."""
    spec = importlib.util.find_spec('edi')
    if spec is None or not spec.origin or not _installed_from(url, digest):
        return False
    dist = importlib.metadata.distribution('easydiffraction')
    origin = os.path.realpath(spec.origin)
    return any(os.path.realpath(dist.locate_file(file)) == origin for file in dist.files or ())


def _pip(*arguments: str, settings: bool = False) -> None:
    """Run pip, ignoring every pip setting unless ``settings`` asks for the user's own."""
    environment = dict(os.environ)
    if not settings:
        environment = {k: v for k, v in environment.items() if not k.startswith('PIP_')}
        environment['PIP_CONFIG_FILE'] = os.devnull
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--only-binary=:all:', *arguments],
        env=environment,
    )

install_edi('https://enhantica.github.io/edi/pypi/simple/')

import io
import os
import tempfile
import urllib.request
import zipfile
from pathlib import Path

if "google.colab" in sys.modules:
    workspace = Path(tempfile.mkdtemp(prefix="edi-tutorial-"))
    (workspace / "notebooks").mkdir()
    os.chdir(workspace / "notebooks")
    target = Path('../cli/pd-neut-cwl_yalo3-al2o3-spodi/project').parent.resolve()
    target.mkdir(parents=True, exist_ok=True)
    url = "https://enhantica.github.io/edi/assets/tutorials/pd-neut-cwl_yalo3-al2o3-spodi.zip"
    with urllib.request.urlopen(url, timeout=60) as response:
        data = response.read()
    with zipfile.ZipFile(io.BytesIO(data)) as bundle:
        for member in bundle.infolist():
            if not (target / member.filename).resolve().is_relative_to(target):
                raise ValueError("Unexpected path in tutorial bundle")
        bundle.extractall(target)


In [ ]:
"""Two-phase fit: YAlO₃ and Al₂O₃, SPODI."""

# Two-phase fit: YAlO₃ and Al₂O₃, SPODI

Fit two phases, YAlO₃ and corundum Al₂O₃, in one constant-wavelength neutron pattern from SPODI
at FRM II, measured at 3 K, with a shared Bérar-Baldinozzi profile and cylinder absorption.

The project is the catalogue case `pd-neut-cwl_yalo3-al2o3-spodi`: the files the command line
fits with `python -m edi fit docs/user/cli/pd-neut-cwl_yalo3-al2o3-spodi/project` and the app
opens from its Examples.

> This tutorial is not run when the documentation is built yet. It loads its project from the edi
> repository, so it runs once a tutorial runner can provide the project elsewhere
> (`execution-exclusions.yml`).

In [ ]:
import edi

## Load the project

In [ ]:
project = edi.Project.load('../cli/pd-neut-cwl_yalo3-al2o3-spodi/project')
print(project.metadata.title)
print('Structures:', ', '.join(structure.name for structure in project.structures))
print('Experiments:', ', '.join(experiment.name for experiment in project.experiments))
print('Fitting mode:', project.fitting_mode)

## The start

The project starts the way your own measurement would: zero shift 0, profile terms to one
significant figure, Biso 0.5 on every site, coordinates to two decimals, a flat background and
the scale to one significant figure.

The fit is free to change these parameters:

In [ ]:
for parameter in project.free_parameters:
    print(f'{parameter.name:28} {parameter.value:12.6g} {parameter.units}')

## Fit

`analysis.fit()` runs the fit the project declares, here one experiment.

In [ ]:
result = project.analysis.fit()
print(f'{result.status.name.lower()} after {result.iterations} iterations')
print(f'reduced chi-square {result.reduced_chi_square:.3f}, Rwp {100 * result.rwp:.2f} %')
for bank in result.banks:
    print(f'  {bank.name}: Rwp {100 * bank.rwp:.2f} %')

## Fitted values

In [ ]:
for label, value in result.values.items():
    print(f'{label:52} {value:12.6g} ± {result.uncertainty.get(label, 0):.2g}')

## Patterns and report

The HTML report draws the measured, calculated and difference patterns with the reflection
positions, and lists the structure and the fit. Open it in a browser.

In [ ]:
project.report.save_html('pd-neut-cwl_yalo3-al2o3-spodi.html')

## Save

`save_as` writes the project with its new values to a folder of its own; the catalogue's copy
stays as it was.

In [ ]:
project.save_as('pd-neut-cwl_yalo3-al2o3-spodi')

## Explore the interactive report

Run the cell and trust the notebook to interact with the report. It includes Plotly plots and three.js structures where available.

In [ ]:
project.report.html_offline = True
project.report